# 12. 비 오는 날의 "택시 블랙홀"

비가 오면 택시 수요는 폭증하지만, 공급(빈차)은 오히려 감소하는 지역이 있다.  
**수요 증가율 - 공급 변화율 = "택시 잡기 힘든 정도" 지수**로 블랙홀 지역을 찾는다.

**데이터:**
- DC_TBYXD012 (승하차 이력, ~6억건)
- seoul_weather_2018_2026.csv (시간별 강수량)

In [ ]:
import gc, psutil, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac: plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

CHUNK_SIZE = 1_000_000
D012_PATH = './DC_TBYXD012.csv'
WEATHER_PATH = './external_data/weather/seoul_weather_2018_2026.csv'

def mem_usage():
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

mem_usage()

## 0. 날씨 데이터 로드 및 비/맑음 분류

In [ ]:
weather = pd.read_csv(WEATHER_PATH)
print('날씨 데이터 컬럼:', weather.columns.tolist())
print(f'날씨 데이터 행수: {len(weather):,}')
weather.head()

In [ ]:
# datetime 파싱 -> date, hour 컬럼 생성
weather['datetime'] = pd.to_datetime(weather['datetime'])
weather['date'] = weather['datetime'].dt.strftime('%Y%m%d')
weather['hour'] = weather['datetime'].dt.hour.astype(np.int8)

# 비 여부 판정: precipitation > 0 (또는 snowfall > 0)
# 컬럼명이 다를 수 있으므로 유연하게 처리
precip_col = [c for c in weather.columns if 'precip' in c.lower()]
snow_col = [c for c in weather.columns if 'snow' in c.lower()]

weather['is_rain'] = False
if precip_col:
    weather['is_rain'] = weather['is_rain'] | (weather[precip_col[0]].fillna(0) > 0)
    print(f'강수 컬럼: {precip_col[0]}')
if snow_col:
    weather['is_rain'] = weather['is_rain'] | (weather[snow_col[0]].fillna(0) > 0)
    print(f'적설 컬럼: {snow_col[0]}')

# 조인 키 생성
weather_lookup = weather[['date', 'hour', 'is_rain']].copy()
weather_lookup['date_hour'] = weather_lookup['date'] + '_' + weather_lookup['hour'].astype(str).str.zfill(2)
rain_set = set(weather_lookup.loc[weather_lookup['is_rain'], 'date_hour'].values)
clear_set = set(weather_lookup.loc[~weather_lookup['is_rain'], 'date_hour'].values)

print(f'비 오는 시간대: {len(rain_set):,}개')
print(f'맑은 시간대: {len(clear_set):,}개')
print(f'비 오는 비율: {len(rain_set)/(len(rain_set)+len(clear_set))*100:.1f}%')

del weather
gc.collect()
mem_usage()

## 1~2. 행정동별 맑은 날/비 오는 날 수요 집계

chunk별로 date+hour 기준 날씨 조인 후 행정동(RIDE_A_CD)별 승차 수요를 집계한다.

In [ ]:
from collections import defaultdict

# 행정동별 수요 (승차 건수)
demand_rain = defaultdict(int)   # RIDE_A_CD -> count (비 올 때)
demand_clear = defaultdict(int)  # RIDE_A_CD -> count (맑을 때)

# 행정동별 공급 (하차 건수 = 빈차 발생)
supply_rain = defaultdict(int)   # ALIGHT_A_CD -> count (비 올 때)
supply_clear = defaultdict(int)  # ALIGHT_A_CD -> count (맑을 때)

# 비/맑은 시간대 수 (날짜 기준 정규화용)
n_rain_hours = len(rain_set)
n_clear_hours = len(clear_set)

cols = ['RIDE_DTIME', 'RIDE_A_CD', 'ALIGHT_A_CD']
dtypes = {'RIDE_DTIME': 'str', 'RIDE_A_CD': 'category', 'ALIGHT_A_CD': 'category'}

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=cols,
                                       dtype=dtypes, chunksize=CHUNK_SIZE)):
    chunk = chunk.dropna(subset=['RIDE_DTIME'])
    
    # date_hour 키 생성
    chunk['date_hour'] = chunk['RIDE_DTIME'].str[:8] + '_' + chunk['RIDE_DTIME'].str[8:10]
    
    # 비/맑음 분류
    chunk['is_rain'] = chunk['date_hour'].isin(rain_set)
    chunk['is_clear'] = chunk['date_hour'].isin(clear_set)
    
    # 비 올 때 수요 (승차)
    rain_df = chunk.loc[chunk['is_rain']]
    if len(rain_df) > 0:
        for code, cnt in rain_df['RIDE_A_CD'].value_counts().items():
            demand_rain[code] += cnt
        for code, cnt in rain_df['ALIGHT_A_CD'].value_counts().items():
            supply_rain[code] += cnt
    
    # 맑을 때 수요 (승차)
    clear_df = chunk.loc[chunk['is_clear']]
    if len(clear_df) > 0:
        for code, cnt in clear_df['RIDE_A_CD'].value_counts().items():
            demand_clear[code] += cnt
        for code, cnt in clear_df['ALIGHT_A_CD'].value_counts().items():
            supply_clear[code] += cnt
    
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
        mem_usage()
    del chunk, rain_df, clear_df
    gc.collect()

print(f'수요 집계 완료 - 행정동 수: {len(demand_rain):,} (비) / {len(demand_clear):,} (맑음)')
mem_usage()

## 3. 행정동별 "비 올 때 수요 증가율" Top 20

In [ ]:
# 시간당 평균으로 정규화
all_codes = set(demand_rain.keys()) | set(demand_clear.keys())

records = []
for code in all_codes:
    d_rain = demand_rain.get(code, 0) / max(n_rain_hours, 1)
    d_clear = demand_clear.get(code, 0) / max(n_clear_hours, 1)
    s_rain = supply_rain.get(code, 0) / max(n_rain_hours, 1)
    s_clear = supply_clear.get(code, 0) / max(n_clear_hours, 1)
    records.append({
        'dong_cd': code,
        'demand_rain_avg': d_rain,
        'demand_clear_avg': d_clear,
        'supply_rain_avg': s_rain,
        'supply_clear_avg': s_clear,
    })

df_dong = pd.DataFrame(records)

# 최소 건수 필터 (시간당 평균 1건 이상)
df_dong = df_dong[(df_dong['demand_clear_avg'] >= 1) & (df_dong['supply_clear_avg'] >= 1)].copy()

# 수요 증가율
df_dong['demand_change_pct'] = (
    (df_dong['demand_rain_avg'] - df_dong['demand_clear_avg'])
    / df_dong['demand_clear_avg'] * 100
)

# 공급 변화율 (하차 = 빈차 발생)
df_dong['supply_change_pct'] = (
    (df_dong['supply_rain_avg'] - df_dong['supply_clear_avg'])
    / df_dong['supply_clear_avg'] * 100
)

print(f'분석 대상 행정동: {len(df_dong):,}개')
df_dong.head()

In [ ]:
# 수요 증가율 Top 20
top20_demand = df_dong.nlargest(20, 'demand_change_pct')

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(top20_demand))
ax.barh(y_pos, top20_demand['demand_change_pct'].values, color='#2196F3', alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(top20_demand['dong_cd'].astype(str).values)
ax.invert_yaxis()
ax.set_xlabel('수요 증가율 (%)')
ax.set_title('비 올 때 수요 증가율 Top 20 행정동')

for i, v in enumerate(top20_demand['demand_change_pct'].values):
    ax.text(v + 0.5, i, f'+{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. 행정동별 "비 올 때 공급 감소율" Top 20

In [ ]:
# 공급 감소가 큰 곳 = supply_change_pct가 가장 낮은 곳
top20_supply = df_dong.nsmallest(20, 'supply_change_pct')

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(top20_supply))
ax.barh(y_pos, top20_supply['supply_change_pct'].values, color='#F44336', alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(top20_supply['dong_cd'].astype(str).values)
ax.invert_yaxis()
ax.set_xlabel('공급 변화율 (%)')
ax.set_title('비 올 때 공급(빈차) 감소율 Top 20 행정동')

for i, v in enumerate(top20_supply['supply_change_pct'].values):
    ax.text(v - 0.5, i, f'{v:.1f}%', va='center', ha='right', fontsize=9)

plt.tight_layout()
plt.show()

## 5. 택시 블랙홀 지수 = 수요 증가율 - 공급 변화율

수요는 크게 늘었는데 공급(빈차)은 줄어든 지역 = 갭이 큰 곳이 블랙홀이다.

In [ ]:
# 블랙홀 지수: 수요 증가율 - 공급 변화율
# 수요가 +30% 늘고, 공급이 -10% 줄면 -> 지수 = 30 - (-10) = 40
df_dong['blackhole_idx'] = df_dong['demand_change_pct'] - df_dong['supply_change_pct']

top10_blackhole = df_dong.nlargest(10, 'blackhole_idx').copy()

print('=== 택시 블랙홀 Top 10 ===')
display_cols = ['dong_cd', 'demand_change_pct', 'supply_change_pct',
                'blackhole_idx', 'demand_rain_avg', 'demand_clear_avg']
print(top10_blackhole[display_cols].to_string(index=False, float_format='{:.1f}'.format))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(top10_blackhole))
labels = top10_blackhole['dong_cd'].astype(str).values

# 수요 증가율 (파란색)
ax.barh(y_pos + 0.15, top10_blackhole['demand_change_pct'].values,
        height=0.3, color='#2196F3', alpha=0.85, label='수요 증가율')

# 공급 변화율 (빨간색)
ax.barh(y_pos - 0.15, top10_blackhole['supply_change_pct'].values,
        height=0.3, color='#F44336', alpha=0.85, label='공급 변화율')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('변화율 (%)')
ax.set_title('택시 블랙홀 Top 10: 수요 증가 vs 공급 감소')
ax.legend(loc='lower right')

# 블랙홀 지수 텍스트
for i, row in enumerate(top10_blackhole.itertuples()):
    ax.text(max(row.demand_change_pct, 0) + 1, i,
            f'GAP: {row.blackhole_idx:.1f}',
            va='center', fontsize=9, fontweight='bold', color='purple')

plt.tight_layout()
plt.show()

In [ ]:
# 전체 행정동 scatter: 수요증가율 vs 공급변화율
fig, ax = plt.subplots(figsize=(10, 10))

ax.scatter(df_dong['supply_change_pct'], df_dong['demand_change_pct'],
           s=10, alpha=0.3, color='gray')

# 블랙홀 Top 10 강조
ax.scatter(top10_blackhole['supply_change_pct'], top10_blackhole['demand_change_pct'],
           s=80, color='red', zorder=5, edgecolors='black', linewidths=0.5)

for _, row in top10_blackhole.iterrows():
    ax.annotate(str(row['dong_cd']),
                (row['supply_change_pct'], row['demand_change_pct']),
                fontsize=7, ha='left', va='bottom')

ax.axhline(0, color='black', lw=0.5)
ax.axvline(0, color='black', lw=0.5)

# 사분면 설명
ax.text(0.02, 0.98, '수요 UP / 공급 UP\n(양호)', transform=ax.transAxes,
        va='top', fontsize=9, color='green')
ax.text(0.98, 0.98, '수요 UP / 공급 DOWN\n(블랙홀)', transform=ax.transAxes,
        va='top', ha='right', fontsize=9, color='red', fontweight='bold')

ax.set_xlabel('공급 변화율 (%)')
ax.set_ylabel('수요 증가율 (%)')
ax.set_title('비 오는 날: 행정동별 수요-공급 변화 (블랙홀 = 우상단)')
plt.tight_layout()
plt.show()

## 6. 블랙홀 지역의 시간대별 패턴

블랙홀 Top 10 지역에서, 비 올 때 특히 몇 시가 최악인지 분석한다.

In [ ]:
# 블랙홀 Top 10 행정동 코드
blackhole_codes = set(top10_blackhole['dong_cd'].astype(str).values)
print(f'블랙홀 행정동 코드: {blackhole_codes}')

# 시간대별 수요 집계 (비/맑음)
# {dong_cd: {hour: count}} 구조
bh_demand_rain_hour = defaultdict(lambda: np.zeros(24, dtype=np.int64))
bh_demand_clear_hour = defaultdict(lambda: np.zeros(24, dtype=np.int64))

cols = ['RIDE_DTIME', 'RIDE_A_CD']
dtypes = {'RIDE_DTIME': 'str', 'RIDE_A_CD': 'str'}

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=cols,
                                       dtype=dtypes, chunksize=CHUNK_SIZE)):
    chunk = chunk.dropna()
    # 블랙홀 지역만 필터
    chunk = chunk[chunk['RIDE_A_CD'].isin(blackhole_codes)]
    if len(chunk) == 0:
        del chunk
        gc.collect()
        continue
    
    chunk['date_hour'] = chunk['RIDE_DTIME'].str[:8] + '_' + chunk['RIDE_DTIME'].str[8:10]
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype(np.int8)
    chunk['is_rain'] = chunk['date_hour'].isin(rain_set)
    chunk['is_clear'] = chunk['date_hour'].isin(clear_set)
    
    for dong_cd, grp in chunk.groupby('RIDE_A_CD'):
        rain_grp = grp.loc[grp['is_rain']]
        clear_grp = grp.loc[grp['is_clear']]
        for h in range(24):
            bh_demand_rain_hour[dong_cd][h] += (rain_grp['hour'] == h).sum()
            bh_demand_clear_hour[dong_cd][h] += (clear_grp['hour'] == h).sum()
    
    if (i + 1) % 100 == 0:
        print(f'  chunk {i+1} 처리 완료')
    del chunk
    gc.collect()

mem_usage()

In [ ]:
# 시간대별 수요 증가율 히트맵
bh_codes_sorted = top10_blackhole['dong_cd'].astype(str).values
hour_change = np.zeros((len(bh_codes_sorted), 24), dtype=np.float32)

for i, code in enumerate(bh_codes_sorted):
    for h in range(24):
        rain_avg = bh_demand_rain_hour[code][h] / max(n_rain_hours / 24, 1)
        clear_avg = bh_demand_clear_hour[code][h] / max(n_clear_hours / 24, 1)
        if clear_avg > 0:
            hour_change[i, h] = (rain_avg - clear_avg) / clear_avg * 100

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(hour_change, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}시' for h in range(24)], rotation=45)
ax.set_yticks(range(len(bh_codes_sorted)))
ax.set_yticklabels(bh_codes_sorted)
ax.set_xlabel('시간대')
ax.set_ylabel('행정동 코드')
ax.set_title('블랙홀 Top 10: 시간대별 수요 증가율 (%, 비 vs 맑음)')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('수요 증가율 (%)')

# 최악 시간대 텍스트
for i in range(len(bh_codes_sorted)):
    worst_h = np.argmax(hour_change[i])
    ax.text(worst_h, i, f'{hour_change[i, worst_h]:.0f}%',
            ha='center', va='center', fontsize=7, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

In [ ]:
# 블랙홀 Top 5 지역의 시간대별 수요 비교 (선 그래프)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, code in enumerate(bh_codes_sorted[:6]):
    ax = axes[idx]
    rain_hourly = np.array([
        bh_demand_rain_hour[code][h] / max(n_rain_hours / 24, 1)
        for h in range(24)
    ])
    clear_hourly = np.array([
        bh_demand_clear_hour[code][h] / max(n_clear_hours / 24, 1)
        for h in range(24)
    ])
    
    hours = np.arange(24)
    ax.plot(hours, clear_hourly, 'o-', color='#2196F3', label='맑은 날', markersize=4)
    ax.plot(hours, rain_hourly, 's-', color='#F44336', label='비 오는 날', markersize=4)
    ax.fill_between(hours, clear_hourly, rain_hourly,
                    where=(rain_hourly > clear_hourly),
                    alpha=0.2, color='red')
    
    worst_h = np.argmax(rain_hourly - clear_hourly)
    ax.axvline(worst_h, ls=':', color='red', alpha=0.5)
    ax.text(worst_h, ax.get_ylim()[1]*0.9 if ax.get_ylim()[1] > 0 else rain_hourly.max()*0.9,
            f'최악: {worst_h}시', fontsize=8, color='red')
    
    ax.set_title(f'행정동 {code}')
    ax.set_xlabel('시간대')
    ax.set_ylabel('시간당 평균 수요')
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=8)

plt.suptitle('블랙홀 지역: 비 오는 날 vs 맑은 날 시간대별 수요', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. 종합 요약

In [ ]:
print('=' * 70)
print('비 오는 날의 "택시 블랙홀" - 종합 요약')
print('=' * 70)
print()
print(f'[분석 기간] 날씨 데이터 기준')
print(f'  - 비 오는 시간대: {n_rain_hours:,}시간 ({n_rain_hours/(n_rain_hours+n_clear_hours)*100:.1f}%)')
print(f'  - 맑은 시간대: {n_clear_hours:,}시간')
print()
print('[택시 블랙홀 Top 10]')
print('  (수요 급증 + 공급 감소 = 택시 잡기 가장 힘든 지역)')
print()
print(f'{"행정동":>12} {"수요증가율":>10} {"공급변화율":>10} {"블랙홀지수":>10} {"최악시간대":>10}')
print('-' * 55)

for _, row in top10_blackhole.iterrows():
    code = str(row['dong_cd'])
    # 최악 시간대 계산
    worst_hours = []
    if code in bh_demand_rain_hour:
        diff = np.array([
            bh_demand_rain_hour[code][h] / max(n_rain_hours/24, 1)
            - bh_demand_clear_hour[code][h] / max(n_clear_hours/24, 1)
            for h in range(24)
        ])
        worst_h = np.argmax(diff)
        worst_hours.append(f'{worst_h}시')
    
    print(f'{code:>12} {row["demand_change_pct"]:>+9.1f}% {row["supply_change_pct"]:>+9.1f}% '
          f'{row["blackhole_idx"]:>9.1f} {" ".join(worst_hours):>10}')

print()
print('[해석]')
print('  - 블랙홀 지수가 높을수록 비 올 때 택시 잡기 어려운 지역')
print('  - 수요 증가율: 비 올 때 승차 수요가 맑은 날 대비 몇 % 증가했는지')
print('  - 공급 변화율: 비 올 때 빈차(하차) 발생이 몇 % 변했는지')
print('  - 블랙홀 지수 = 수요증가율 - 공급변화율 (갭이 클수록 심각)')
print()
print('=' * 70)
mem_usage()